# Thermal comfort and urban heat
## Mexicali Urban Liveability Index — `WP02_thermal_comfort_and_heat`

**Lead:** Rossano Schifanella
**Indicators assigned:** 7
**Schema version:** 1.0.0

Outdoor thermal comfort and urban heat exposure for an arid city: temperature, humidity, sunshine/shade, departures from human comfort optima, and the urban heat island.

Cross-check against the GHSCI Global Urban Heat Vulnerability Index (GUHVI) layers already produced for Mexicali, and agree which is authoritative before duplicating effort.

> New to this project? Work through
> [`00_overview_and_schema.ipynb`](00_overview_and_schema.ipynb)
> first — it carries one indicator end to end. Then read
> [`docs/analyst_guide.md`](../docs/analyst_guide.md).

## How to work through this notebook

For each indicator assigned to you, in this order:

1. **Read the brief.** It reproduces everything the team already
   recorded in the workbook — the draft rationale, the article the
   indicator was adapted from, candidate data sources, and the open
   questions colleagues raised. Do not retype any of it; it is
   already in your metadata stub.
2. **Write the causal pathway sentence** (guide §2.1) and find
   **independent health evidence** for it (§2.2). Do this *before*
   looking for data. Fill in `meta['rationale']`.
3. **Find and document the data** (§3): citation, URL, date
   retrieved, licence, and whether it reaches Condesa.
4. **Compute** at the finest scale your data genuinely support.
   Produce a `DataFrame` with `geo_id` and `value`.
5. **Harmonise** with `uli.harmonise(...)`, label with
   `uli.label(...)`, and **deliver** with
   `uli.write_indicator(...)`.
6. **Look at the map.** Most errors are obvious in ten seconds and
   invisible in a table.

`uli.write_indicator` validates first and refuses to publish a
failing deliverable. While you are still iterating, pass
`allow_failure=True` to write a draft anyway.

Full guidance: [`docs/analyst_guide.md`](../docs/analyst_guide.md).
Schema: [`schema/ULI_output_schema.md`](../schema/ULI_output_schema.md).

## Framing the indicator against health evidence

Every indicator must be justified by evidence of a **meaningful
health or wellbeing benefit**, independent of the liveability
article it was adapted from. Those articles establish that an
indicator is used; they rarely establish that it matters.

Complete this sentence before you compute anything:

> *[what I measure]* changes *[a mechanism]*, which changes *[a
> behaviour or exposure]*, which affects *[a health outcome]*.

For most indicators in this project the behaviour is **walking for
transport**, **walking or recreation in public space**, or
**social contact** — and the exposure is **heat**, **air
pollution** or **injury risk**. Say which, using the vocabulary in
`uli.vocab.HEALTH_PATHWAYS`.

Prefer meta-analyses and systematic reviews, then reputable
guidance (WHO, UN-Habitat, PAHO, Secretaría de Salud), then cohort
studies and natural experiments. Record the **effect size with its
uncertainty**.

**If the evidence supports a different threshold from the one the
workbook proposes, use the evidence-based threshold** and say so in
`threshold_justification`. That is explicitly what the project
wants.

**Mexicali is arid and extremely hot.** Most of this literature
comes from temperate cities. Where the transfer is doubtful — for
example, distance-based walkability thresholds in a city where
summer maxima exceed 45 °C and shade rather than distance is the
binding constraint — record it in `rationale.arid_context`. That is
a contribution, not a caveat.

## When several workbook rows are really one indicator

The workbook harvested indicators article by article, so a single
construct sometimes appears as several rows seen through different
lenses or over different time periods. Air quality is the clearest
case:

| Row | What it is | Lens | Time basis |
|---|---|---|---|
| #292 Air quality | the index value itself | `quality` | `annual_mean` |
| #8 Good air quality | that value against a standard | `quality` | `threshold_share` |
| #293 Days with good air quality | how often the standard is met | `quantity` | `threshold_compliance_days` |
| #173 Days PM2.5 over WHO | the same, for one pollutant | `quantity` | `threshold_exceedance_days` |

These are not four indicators — they are one construct measured
four ways, and computing them separately would mean four
inconsistent methods and four sets of data documentation.

Deliver them as a **measure family**: give every measure the same
`measure_family` slug, and distinguish them with `temporal_basis`
(see `uli.vocab.TEMPORAL_BASES`) and `threshold`. They can still
live under separate workbook ids — the family slug is what tells
the index step, and Reimagina Urbana, that they belong together.

```python
for meta in (meta_292, meta_8, meta_293, meta_173):
    for measure in meta['measures']:
        measure['measure_family'] = 'air_quality'
    meta['data_sources'] = SHARED_SOURCES   # one method, one source
```

The same pattern applies to mean summer temperature versus days
above a comfort threshold (WP02), and to flood extent versus annual
average days of flooding (WP05).

## Condesa coverage is a requirement, not a nicety

The Condesa new development in south-east Mexicali is a project
focus area, and it defeats the usual assumptions:

- about **20%** of it falls outside the previously configured
  study region boundary;
- only **44%** of its area is covered by census manzana polygons,
  so a **manzana-native calculation reaches 33 of the 40
  fraccionamientos, while a `grid_100m`-native one reaches all
  40**;
- it is platted and roaded (43 km of street network in OpenStreetMap
  across 27 of the 40 fraccionamientos) but essentially unbuilt —
  **zero destinations**, and satellite-derived population products
  see almost nobody there.

**One thing is your decision: the native scale.** If your data
allow it, compute on the 100 m grid. That is the difference between
reaching all of Condesa and quietly missing a fifth of it.

Everything else is handled downstream. Population denominators,
the 2030 occupancy scenario and population-weighted exposure
statistics are a reporting-step concern (`uli.exposure`), decided
once for the whole project rather than by each analyst. Urban
fabric and exposure measures — land cover, air quality, heat,
hazards, street infrastructure — are properties of *place*, and
should be computed as such; who lives there is applied later.

Two things to record, though:

- `data_sources[].condesa_coverage` — whether your **source**
  reaches Condesa. Satellite imagery and OSM generally do; a 2020
  census variable or a household survey generally does not.
- `method.condesa_treatment` — what you did about it. Where a
  source does not reach Condesa, mark those rows `no_data` rather
  than omitting them.

The validator treats poor Condesa coverage as an **error**.

---
## Setup

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uli

# Identify yourself once; it is copied into every deliverable.
ANALYST = {
    'name': 'TODO: your name',
    'email': None,
    'institution': None,
}

print(f'ULI schema version {uli.SCHEMA_VERSION}')
print(f'Reference geographies available: {uli.geography.available()}')

        WORK_PACKAGE = 'WP02_thermal_comfort_and_heat'
        NOTEBOOK = 'notebooks/02_thermal_comfort_and_heat.ipynb'

---
## Your indicators (7)

Each has a brief reproducing what the workbook records,
then three working cells: documentation, calculation,
delivery.

### 382 — Urban heat island

`urban_heat_island` · *Ambient Environment · Risks · Heat · Urban heat island*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Mitigating the urban heat island effect by reducing localized temperature increases promotes thermal comfort and supports the physical and mental well-being of city dwellers.
- **Adapted from:** Leya et al., 2022, 'Spatial Variations of Urban Heat Island Development in Khulna City, Bangladesh: Implications for Urban Planning and Development'
- **Effect reported there:** Association reported; no effect size provided.
- **Methods used in the literature:** ['Analytic Hierarchy Process (AHP)']
- **Open questions raised:** ER: Key indicator

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_382) at any time to list what is
# still outstanding.
meta_382 = uli.metadata_stub(382, analyst=ANALYST)

# meta_382['rationale']['statement'] = """..."""
# meta_382['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_382['rationale']['arid_context'] = '...'
# meta_382['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_382['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_382)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_382 = 'grid_100m'
METHOD_382 = 'population_weighted_mean'

native_382 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_382 = uli.harmonise(
    native_382,
    native_scale=NATIVE_SCALE_382,
    method=METHOD_382,
)
results_382 = uli.label(
    harmonised_382,
    meta_382,
    measure_id='urban_heat_island__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_382, meta_382))
# uli.write_indicator(results_382, meta_382)

### 107 — The difference between the relative humidity in July and the optimum humidity for human body

`the_difference_between_the_relative_humidity_in_july` · *Ambient Environment · Ambient · Thermal comfort · The difference between the relative humidity in July and the optimum humidity for human body*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** The difference from optimal humidity in July affects thermal comfort and respiratory wellness, with extreme humidity levels impacting residents' physical health.
- **Adapted from:** Zhan et al., 2018, 'Assessment and determinants of satisfaction with urban livability in China'; Tang et al., 2017, 'Comprehensive evaluation of trends in human settlements quality changes and spatial differentiation characteristics of 35 Chinese major cities'
- **Effect reported there:** No health outcome assessed.
- **Methods used in the literature:** ['Not specified']
- **Open questions raised:** checar indicator

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_107) at any time to list what is
# still outstanding.
meta_107 = uli.metadata_stub(107, analyst=ANALYST)

# meta_107['rationale']['statement'] = """..."""
# meta_107['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_107['rationale']['arid_context'] = '...'
# meta_107['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_107['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_107)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_107 = 'grid_100m'
METHOD_107 = 'population_weighted_mean'

native_107 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_107 = uli.harmonise(
    native_107,
    native_scale=NATIVE_SCALE_107,
    method=METHOD_107,
)
results_107 = uli.label(
    harmonised_107,
    meta_107,
    measure_id='the_difference_between_the_relative_humidity_in_july__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_107, meta_107))
# uli.write_indicator(results_107, meta_107)

### 337 — Humidity

`humidity` · *Ambient Environment · Ambient · Thermal comfort · Humidity*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Appropriate humidity levels are necessary for human thermal comfort and can affect respiratory health and the perceived quality of the urban environment.
- **Adapted from:** Kashi et al., 2024, 'Effects of extreme weather events and climate change on cities’ livability'; Liang and Li, 2020, 'Resilience and sustainable development goals based social-ecological indicators and assessment of coastal urban areas'
- **Effect reported there:** Association reported; no effect size provided.
- **Pragmatic approach agreed:** Thermal comfort indicator to be calculated by Rossano
- **Methods used in the literature:** ['Exploratory Factor Analysis', 'Standard Deviational Ellipse,', 'Hot Spot Analysis,', 'Network Analysis']
- **Team notes:** ER: Rossano will work on the Thermal comfort indicator

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_337) at any time to list what is
# still outstanding.
meta_337 = uli.metadata_stub(337, analyst=ANALYST)

# meta_337['rationale']['statement'] = """..."""
# meta_337['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_337['rationale']['arid_context'] = '...'
# meta_337['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_337['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_337)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_337 = 'grid_100m'
METHOD_337 = 'population_weighted_mean'

native_337 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_337 = uli.harmonise(
    native_337,
    native_scale=NATIVE_SCALE_337,
    method=METHOD_337,
)
results_337 = uli.label(
    harmonised_337,
    meta_337,
    measure_id='humidity__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_337, meta_337))
# uli.write_indicator(results_337, meta_337)

### 366 — Hours of sunshine in January

`hours_of_sunshine_in_january` · *Ambient Environment · Ambient · Thermal comfort · Hours of sunshine in January*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Adequate sunshine duration in January supports mental wellbeing and provides natural lighting and heating, contributing to a pleasant living environment.
- **Adapted from:** Arpan & Joy, 2018, 'Livability assessment within a metropolis based on the impact of integrated urban geographic factors (IUGFs) on clustering urban centers of Kolkata'; Li & Jin, 2012, 'Characteristics and spatial-temporal differences of urban human settlement environment in China'
- **Effect reported there:** No health outcome assessed.
- **Pragmatic approach agreed:** Thermal comfort indicator to be calculated by Rossano
- **Methods used in the literature:** ['Not specified']
- **Team notes:** ER: Rossano will work on the Thermal comfort indicator
- **Open questions raised:** CH: "Sunshine in January" is a very parochially phrased indicator (assumption that January is in northern hemisphere, so this is sunshine in Winter I assume; or vice versa. But it would be better to be explicit as to which it means!) / ER: This one is part of a "Natural Environment Pleaseantness" I think this could be usueful or it can be part of the heat risk indicators

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_366) at any time to list what is
# still outstanding.
meta_366 = uli.metadata_stub(366, analyst=ANALYST)

# meta_366['rationale']['statement'] = """..."""
# meta_366['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_366['rationale']['arid_context'] = '...'
# meta_366['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_366['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_366)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_366 = 'grid_100m'
METHOD_366 = 'population_weighted_mean'

native_366 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_366 = uli.harmonise(
    native_366,
    native_scale=NATIVE_SCALE_366,
    method=METHOD_366,
)
results_366 = uli.label(
    harmonised_366,
    meta_366,
    measure_id='hours_of_sunshine_in_january__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_366, meta_366))
# uli.write_indicator(results_366, meta_366)

### 370 — Average temperature in sunshine

`average_temperature_in_sunshine` · *Ambient Environment · Ambient · Thermal comfort · Average temperature in sunshine*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Average winter temperature (as January mean) is a key climate amenity that influences residents' physiological comfort and health during the cold season.
- **Adapted from:** Zhan et al., 2018, 'Assessment and determinants of satisfaction with urban livability in China'; Li et al., 2008, 'Review of the theories and methods of livable city'
- **Effect reported there:** No health outcome assessed.
- **Pragmatic approach agreed:** Thermal comfort indicator to be calculated by Rossano
- **Methods used in the literature:** ['Not specified']
- **Team notes:** ER: Rossano will work on the Thermal comfort indicator
- **Open questions raised:** Will this be surface temperature or street level temperature?
- **Feasibility flag:** to be determined

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_370) at any time to list what is
# still outstanding.
meta_370 = uli.metadata_stub(370, analyst=ANALYST)

# meta_370['rationale']['statement'] = """..."""
# meta_370['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_370['rationale']['arid_context'] = '...'
# meta_370['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_370['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_370)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_370 = 'grid_100m'
METHOD_370 = 'population_weighted_mean'

native_370 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_370 = uli.harmonise(
    native_370,
    native_scale=NATIVE_SCALE_370,
    method=METHOD_370,
)
results_370 = uli.label(
    harmonised_370,
    meta_370,
    measure_id='average_temperature_in_sunshine__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_370, meta_370))
# uli.write_indicator(results_370, meta_370)

### 371 — Temperature

`temperature` · *Ambient Environment · Ambient · Thermal comfort · Temperature*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** Maintaining urban temperatures within a comfortable range is vital for physical health and preventing heat-related illnesses and thermal discomfort.
- **Adapted from:** Kashi et al., 2024, 'Effects of extreme weather events and climate change on cities’ livability'; Alijani et al., 2020, 'A new approach of urban livability in Tehran: Thermal comfort as a primitive indicator'
- **Effect reported there:** Association reported; no effect size provided.
- **Pragmatic approach agreed:** Thermal comfort indicator to be calculated by Rossano
- **Methods used in the literature:** ['Exploratory Factor Analysis', 'Standard Deviational Ellipse,', 'Hot Spot Analysis,', 'Network Analysis']
- **Team notes:** ER: Rossano will work on the Thermal comfort indicator

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_371) at any time to list what is
# still outstanding.
meta_371 = uli.metadata_stub(371, analyst=ANALYST)

# meta_371['rationale']['statement'] = """..."""
# meta_371['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_371['rationale']['arid_context'] = '...'
# meta_371['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_371['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_371)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_371 = 'grid_100m'
METHOD_371 = 'population_weighted_mean'

native_371 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_371 = uli.harmonise(
    native_371,
    native_scale=NATIVE_SCALE_371,
    method=METHOD_371,
)
results_371 = uli.label(
    harmonised_371,
    meta_371,
    measure_id='temperature__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_371, meta_371))
# uli.write_indicator(results_371, meta_371)

### 372 — The difference between the average temperature in July and the optimal temperature for human body

`the_difference_between_the_average_temperature_in_july` · *Ambient Environment · Ambient · Thermal comfort · The difference between the average temperature in July and the optimal temperature for human body*

- **Lenses to deliver:** quantity
- **Draft rationale (rewrite this):** The difference from optimal temperature in July indicates thermal comfort; excessive summer heat is linked to physiological stress and reduced outdoor activity.
- **Adapted from:** Zhan et al., 2018, 'Assessment and determinants of satisfaction with urban livability in China'; Cao et al., 2019, 'The framework of relationship between built environment and residents healthy based on activity perspective'
- **Effect reported there:** No health outcome assessed.
- **Pragmatic approach agreed:** Thermal comfort indicator to be calculated by Rossano
- **Methods used in the literature:** ['Not specified']
- **Team notes:** ER: Rossano will work on the Thermal comfort indicator
- **Open questions raised:** Care needs to be taken: is "July" relevant for Mexicali? (e.g. better to calculate as "Hottest month" if that was intended by the authors; July is too parochial)

> **Your first task is not GIS.** Write the causal pathway sentence, then find independent health evidence for it — see the framing section above.

In [ ]:
# 1. Documentation --------------------------------------------
# Pre-filled from the workbook; edit in place.  Run
# uli.todos(meta_372) at any time to list what is
# still outstanding.
meta_372 = uli.metadata_stub(372, analyst=ANALYST)

# meta_372['rationale']['statement'] = """..."""
# meta_372['rationale']['health_pathways'] = [
#     'physical_activity_transport',
# ]
# meta_372['rationale']['arid_context'] = '...'
# meta_372['rationale']['evidence'] = [
#     {
#         'claim': '...',
#         'citation': '...',
#         'doi': '...',
#         'evidence_type': 'systematic_review',
#         'population': '...',
#         'exposure': '...',
#         'outcome': '...',
#         'effect': 'RR 0.92 (95% CI 0.88-0.96) per ...',
#         'threshold_support': '...',
#     },
# ]
# meta_372['data_sources'] = [
#     {
#         'name': '...',
#         'custodian': '...',
#         'citation': '...',
#         'url': '...',
#         'date_retrieved': 'YYYY-MM-DD',
#         'licence': '...',
#         'redistributable': True,
#         'spatial_resolution': '...',
#         'temporal_coverage': '...',
#         'condesa_coverage': 'full',
#     },
# ]

uli.todos(meta_372)

In [ ]:
# 2. Calculation ----------------------------------------------
# Compute at the finest scale your data genuinely support and
# produce a DataFrame with columns: geo_id, value.
#
# Reaching all 40 Condesa fraccionamientos needs a native
# scale of grid_100m (manzana reaches only 33).
NATIVE_SCALE_372 = 'grid_100m'
METHOD_372 = 'population_weighted_mean'

native_372 = None  # TODO: your calculation

# Handy builders:
#   uli.count_features(points, NATIVE_SCALE, per='1000_persons')
#   uli.areal_share(polygons, NATIVE_SCALE, as_percentage=True)
#   uli.network_share(edges, NATIVE_SCALE, 'has_sidewalk')
#   uli.zonal_statistic('raster.tif', NATIVE_SCALE, 'mean')

In [ ]:
# 3. Harmonise, validate, deliver ------------------------------
harmonised_372 = uli.harmonise(
    native_372,
    native_scale=NATIVE_SCALE_372,
    method=METHOD_372,
)
results_372 = uli.label(
    harmonised_372,
    meta_372,
    measure_id='the_difference_between_the_average_temperature_in_july__quantity',
)
# Several measures?  Label each and combine:
#   results = uli.assemble([results_a, results_b])

print(uli.check(results_372, meta_372))
# uli.write_indicator(results_372, meta_372)

---
## Check what this work package has delivered

In [ ]:
delivered, catalogue = uli.collect()
if len(catalogue):
    display(catalogue)
    print(delivered.groupby(['indicator_code', 'geo_level']).size())
else:
    print('Nothing delivered yet.')

In [ ]:
# Sanity-check a delivered measure on a map before you call it done.
# MEASURE = 'your_indicator_code__quantity'
# LEVEL = 'manzana'
# units = uli.geography.load(LEVEL).merge(
#     delivered.query('measure_id == @MEASURE and geo_level == @LEVEL'),
#     on='geo_id', how='left')
# ax = units.plot(column='value', legend=True, figsize=(11, 8),
#                 missing_kwds={'color': 'lightgrey'})
# condesa = uli.geography.load('condesa_fraccionamiento')
# condesa.boundary.plot(ax=ax, color='red', linewidth=1)
# ax.set_title(MEASURE)
# ax.set_axis_off()